# Fase 5 — Evaluation | CardioRisk · CRISP-DM
Evaluación clínica sobre **TEST SET SELLADO**.
El test set no fue visto durante entrenamiento ni optimización en F4.

In [ ]:
# BLOQUE 1 — INSTALACIONES + CARGA + PIPELINE
!pip install -q kagglehub imbalanced-learn xgboost shap

import warnings, joblib
import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import shap
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (recall_score, precision_score, f1_score, roc_auc_score,
    accuracy_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve, auc)
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'figure.facecolor': '#0a0f1a', 'axes.facecolor': '#0d1526',
    'axes.edgecolor': '#1a2c3d',   'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',       'xtick.color': '#7a8fa8',
    'ytick.color': '#7a8fa8',      'grid.color': '#1a2c3d',
    'savefig.facecolor': '#0a0f1a'
})

try:
    %store -r X_train_sm
    %store -r y_train_sm
    %store -r X_val_sc
    %store -r y_val
    %store -r X_test_sc
    %store -r y_test
    %store -r scaler
    %store -r FEATURES_FINALES
    print(f'✓ Artefactos cargados — {len(FEATURES_FINALES)} features')
except:
    print('⚠ %store no disponible — ejecutando CANON completo')
    import kagglehub, os
    try:
        path = kagglehub.dataset_download('jocelyndumlao/cardiovascular-disease-dataset')
        csv_path = next(os.path.join(r,f) for r,_,fs in os.walk(path) for f in fs if f.endswith('.csv'))
        df = pd.read_csv(csv_path)
    except:
        df = pd.read_csv('/content/cardiovascular_disease_dataset.csv')
    df.columns = df.columns.str.lower().str.strip()
    df = df.dropna()
    df['slope_x_oldpeak'] = df['slope'] * df['oldpeak']
    df['log_oldpeak']     = np.log1p(df['oldpeak'])
    CATEGORICAL = ['gender', 'chestpain', 'restingrelectro']
    TARGET = 'target'
    df_enc = pd.get_dummies(df, columns=CATEGORICAL, drop_first=False)
    EXCLUIR = [TARGET, 'patientid']
    FEATURES_FINALES = [c for c in df_enc.columns if c not in EXCLUIR]
    X = df_enc[FEATURES_FINALES]; y = df_enc[TARGET]
    X_temp,X_test,y_temp,y_test = train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)
    X_train,X_val,y_train,y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42,stratify=y_temp)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_val_sc   = scaler.transform(X_val)
    X_test_sc  = scaler.transform(X_test)
    sm = SMOTE(random_state=42)
    X_train_sm, y_train_sm = sm.fit_resample(X_train_sc, y_train)

# Modelo con hiperparámetros óptimos de F4
best_model = GradientBoostingClassifier(learning_rate=0.1, max_depth=5, n_estimators=200, random_state=42)
best_model.fit(X_train_sm, y_train_sm)
y_pred = best_model.predict(X_test_sc)
y_prob = best_model.predict_proba(X_test_sc)[:, 1]
print('✓ Modelo entrenado sobre train — evaluando sobre TEST SET SELLADO')


In [ ]:
# BLOQUE 2 — MÉTRICAS COMPLETAS TEST SET
recall       = recall_score(y_test, y_pred)
precision    = precision_score(y_test, y_pred)
f1           = f1_score(y_test, y_pred)
auc_roc      = roc_auc_score(y_test, y_prob)
accuracy     = accuracy_score(y_test, y_pred)
cm           = confusion_matrix(y_test, y_pred)
tn,fp,fn,tp  = cm.ravel()
especificidad = tn/(tn+fp)
error_tipo_ii = fn/(tp+fn) if (tp+fn)>0 else 0

print('='*55)
print('  RESULTADOS FINALES — TEST SET SELLADO')
print('='*55)
print(f'  Recall (Sensibilidad) : {recall:.4f}  ← MÉTRICA PRIMARIA')
print(f'  Precision             : {precision:.4f}')
print(f'  F1-Score              : {f1:.4f}')
print(f'  AUC-ROC               : {auc_roc:.4f}')
print(f'  Accuracy              : {accuracy:.4f}')
print(f'  Especificidad         : {especificidad:.4f}')
print('-'*55)
print(f'  TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print(f'  Error tipo II         : {error_tipo_ii:.4f}')
print('-'*55)
kpi1 = '✓' if recall > 0.80 else '✗'
kpi2 = '✓' if auc_roc > 0.85 else '✗'
kpi3 = '✓' if error_tipo_ii < 0.05 else '✗'
print(f'  {kpi1} KPI Recall > 0.80   : {recall:.4f}')
print(f'  {kpi2} KPI AUC-ROC > 0.85 : {auc_roc:.4f}')
print(f'  {kpi3} KPI Error II < 5%  : {error_tipo_ii:.4f}')
print('='*55)
print(classification_report(y_test, y_pred, target_names=['Bajo Riesgo','Alto Riesgo']))


In [ ]:
# BLOQUE 3 — MATRIZ DE CONFUSIÓN
fig, ax = plt.subplots(figsize=(6, 5))
img = ax.imshow(cm, interpolation='nearest', cmap='Blues', alpha=0.6)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Bajo Riesgo','Alto Riesgo']); ax.set_yticklabels(['Bajo Riesgo','Alto Riesgo'])
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión — Test Set', fontsize=13, fontweight='bold')
labels  = [['TN','FP'],['FN ✗','TP ✓']]
c_text  = [['#00f2fe','#00f2fe'],['#ff3355','#00ff88']]
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=22, fontweight='bold', color=c_text[i][j])
        ax.text(j, i+0.35, labels[i][j], ha='center', va='center', fontsize=9, color='#7a8fa8')
plt.colorbar(img, ax=ax)
plt.tight_layout()
plt.savefig('/content/f5_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# BLOQUE 4 — CURVAS ROC Y PRECISION-RECALL
from sklearn.metrics import average_precision_score

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_test, y_prob)
ax1.plot(fpr, tpr, color='#00ff88', lw=2, label=f'AUC = {auc_roc:.4f}')
ax1.plot([0,1],[0,1], color='#7a8fa8', lw=1, linestyle='--')
ax1.fill_between(fpr, tpr, alpha=0.15, color='#00ff88')
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('Curva ROC', fontsize=13, fontweight='bold')
ax1.legend(facecolor='#0d1526', edgecolor='#1a2c3d', labelcolor='#e2e8f0')
ax1.grid(alpha=0.3)

prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
ax2.plot(rec_c, prec_c, color='#8b5cf6', lw=2, label=f'AP = {ap:.4f}')
ax2.fill_between(rec_c, prec_c, alpha=0.15, color='#8b5cf6')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Curva Precision-Recall', fontsize=13, fontweight='bold')
ax2.legend(facecolor='#0d1526', edgecolor='#1a2c3d', labelcolor='#e2e8f0')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/f5_curves.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# BLOQUE 5 — SHAP FEATURE IMPORTANCE
try:
    explainer  = shap.TreeExplainer(best_model)
    shap_vals  = explainer.shap_values(X_test_sc)
    mean_shap  = np.abs(shap_vals).mean(axis=0)
    top_idx    = np.argsort(mean_shap)[-12:]
    feat_names = list(FEATURES_FINALES)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh([feat_names[i] for i in top_idx], mean_shap[top_idx], color='#8b5cf6', alpha=0.8)
    ax.set_xlabel('|SHAP value| promedio')
    ax.set_title('Importancia de Variables (SHAP) — Top 12', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/f5_shap.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✓ SHAP OK — Top predictor: {feat_names[top_idx[-1]]}')
except Exception as e:
    print(f'SHAP falló ({e}) — usando feature_importances_')
    imp = pd.Series(best_model.feature_importances_, index=FEATURES_FINALES).sort_values()
    fig, ax = plt.subplots(figsize=(10, 6))
    imp[-12:].plot(kind='barh', ax=ax, color='#8b5cf6', alpha=0.8)
    ax.set_title('Importancia de Variables (Gini) — Top 12', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/f5_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

print('→ F5 COMPLETO')
